# Context experiments (ICLR 2027 rebuttal, issues 3 and 7)

This notebook reads the files that five scripts in `scripts/evaluation/` write and turns them into the tables and plots of the six context experiments. It is for inspection. The figures of the paper are drawn in the notebooks that own them.

| Section | Script | Files under `data/analysis/` |
|---|---|---|
| Context-length matrix | `context_length.py` | `context_length/cubic_1d.hdf5`, `context_length/sine_rescaled_<bin>_grf.hdf5` |
| Error against the CFL number | `context_length.py` | `context_length/cubic_1d_cfl.hdf5` |
| Flux refreshed every n steps | `flux_refresh.py` | `flux_refresh/cubic_1d.hdf5` |
| Shuffled, static and donor contexts | `context_interventions.py` | `context_interventions/cubic_1d.hdf5` |
| Effective flux | `effective_flux.py` | `effective_flux/cubic_1d_limit10.hdf5`, `effective_flux/cubic_1d.hdf5`, `effective_flux/sine_rescaled_<bin>_grf.hdf5` |
| One step of another size | `time_step.py` | `time_step/cubic_1d.hdf5` |

A section whose file does not exist prints the missing path and draws nothing, so the notebook runs while the results are still coming in.

Every error is averaged over the segments with a finite value, per seed. A table cell shows the mean over the seeds, and an error bar shows the standard deviation over the seeds.

In [ ]:
from pathlib import Path

import colorcet as cc
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from context_flux_no.evaluation import (
    equal_count_bins,
    FLUX_FAMILIES,
    INTERVENTIONS,
    mean_by_bin,
)
from IPython.display import display
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter


plt.style.use("context_flux_no.plotting.styles.dash_gridded")

analysis_dir = Path("../../data/analysis")

# Group names of the result files -> names and colors of the paper.
LABELS = {"HyperFluxFNOLocal": "HFluxNO", "DPOT": "DPOT", "DISCO": "DISCO"}
COLORS = {
    "HyperFluxFNOLocal": cc.cm.glasbey_bw(41),
    "DPOT": cc.cm.glasbey_bw(85),
    "DISCO": cc.cm.glasbey_bw(68),
}
HFLUXNO = "HyperFluxFNOLocal"
# Neutral colors for the true flux, for references and for annotations.
INK = "0.15"
MUTED = "0.55"
# The four bins of |b| of the rescaled sine sets, in increasing order.
SINE_BINS = ("0-1", "1-2", "2-3", "3-4")
# One hue from light to dark for a quantity that has an order, such as |b|.
ORDERED = cc.cm.blues


def ordered_colors(count: int) -> list:
    """Return `count` steps of one hue, from light to dark."""
    return [ORDERED(level) for level in np.linspace(0.35, 1.0, count)]


def plain_log_x(ax: plt.Axes) -> None:
    """Give a logarithmic x-axis ticks at 1, 2 and 5 times the powers of ten, written
    as plain numbers. It suits an axis that spans about one or two decades."""
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())


def open_result(experiment: str, name: str) -> h5py.File | None:
    """Open one result file. A missing file is printed and gives `None`."""
    path = analysis_dir / experiment / f"{name}.hdf5"
    if not path.exists():
        print(f"Missing {path}")
        return None
    return h5py.File(path, "r")


def mean_over_finite(values: np.ndarray, axis: tuple[int, ...]) -> np.ndarray:
    """Average over `axis`, leaving out the entries that are not finite."""
    is_finite = np.isfinite(values)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(is_finite, values, 0.0).sum(axis=axis) / is_finite.sum(
            axis=axis
        )


def mean_std(per_seed: np.ndarray) -> str:
    """Format the mean and the standard deviation over the seeds, the first axis."""
    return f"{per_seed.mean(axis=0):.3e} ± {per_seed.std(axis=0):.1e}"


def binned_ratio(
    numerator: np.ndarray,
    denominator: np.ndarray,
    bin_indices: np.ndarray,
    num_bins: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Return, per seed and bin, the mean of `numerator` over the mean of
    `denominator`, and the number of entries per bin.

    Both arrays have the seeds as their first axis and are flattened behind it.
    """
    num_seeds = numerator.shape[0]
    top, counts = mean_by_bin(numerator.reshape(num_seeds, -1), bin_indices, num_bins)
    bottom, _ = mean_by_bin(denominator.reshape(num_seeds, -1), bin_indices, num_bins)
    with np.errstate(invalid="ignore", divide="ignore"):
        return top / bottom, counts


def bin_centers(
    values: np.ndarray, bin_indices: np.ndarray, num_bins: int
) -> np.ndarray:
    """Return the median of the values of every bin, NaN for an empty bin."""
    return np.asarray(
        [
            np.median(values[bin_indices == b]) if np.any(bin_indices == b) else np.nan
            for b in range(num_bins)
        ]
    )

## Context-length matrix (E1)

Every checkpoint is evaluated at every test context length. The last context frame is the same for all lengths, so all of them predict the same frames. A row is one training length, and the diagonal holds the models at the length that they were trained on.

`l2_onestep` is the error of one step. `l2_rollout` is the 20-step rollout of the submission, which re-encodes the window at every step. In that rollout a longer window holds true frames for longer, so its gain along the diagonal mixes the context length with a delayed feedback of predictions.

In [ ]:
MATRIX_SETS = {"cubic_1d": "In distribution"} | {
    f"sine_rescaled_{b}_grf": f"Sine, |b| in ({b.replace('-', ', ')}]"
    for b in SINE_BINS
}
MATRIX_FIELDS = {"l2_onestep": "One step", "l2_rollout": "Sliding 20-step rollout"}


def load_matrix(name: str) -> dict | None:
    """Load the per-seed means of one test set.

    The result holds, per field, the mean error of shape
    `(training lengths, seeds, test lengths)` and the mean error of the identity
    baseline over the same segments.
    """
    f = open_result("context_length", name)
    if f is None:
        return None
    with f:
        groups = sorted(f["models"], key=lambda group: int(group.rsplit("_", 1)[1]))
        result = {
            "trained_at": [int(group.rsplit("_", 1)[1]) for group in groups],
            "tested_at": [int(k) for k in f.attrs["test_context_lengths"]],
        }
        for field in MATRIX_FIELDS:
            identity = f[f"baselines/identity/{field}"][()]
            errors, baselines = [], []
            for group in groups:
                values = f[f"models/{group}/{field}"][()]
                is_finite = np.isfinite(values) & np.isfinite(identity)
                values = np.where(is_finite, values, np.nan)
                errors.append(mean_over_finite(values, axis=(2, 3)))
                baselines.append(
                    mean_over_finite(np.where(is_finite, identity, np.nan), axis=(2, 3))
                )
            result[field] = np.stack(errors)
            result[f"{field}_identity"] = np.stack(baselines)
    return result


matrices = {name: load_matrix(name) for name in MATRIX_SETS}
matrices = {name: matrix for name, matrix in matrices.items() if matrix is not None}

### Tables

One table per test set and score. A cell is the mean error with its standard deviation over the seeds and, in parentheses, the error relative to the identity baseline.

In [ ]:
for name, matrix in matrices.items():
    for field, title in MATRIX_FIELDS.items():
        relative = matrix[field] / matrix[f"{field}_identity"]
        table = pd.DataFrame(
            [
                [
                    f"{mean_std(matrix[field][i, :, j])} "
                    f"({relative[i, :, j].mean():.2f})"
                    for j in range(len(matrix["tested_at"]))
                ]
                for i in range(len(matrix["trained_at"]))
            ],
            index=pd.Index(matrix["trained_at"], name="trained at"),
            columns=pd.Index(matrix["tested_at"], name="tested at"),
        )
        print(f"{MATRIX_SETS[name]}: {title}, {matrix[field].shape[1]} seeds")
        display(table)

### The whole matrix

Each panel is one test set, and a cell is the error relative to the identity baseline. A value of 1 means that the model is no better than holding the last context frame. The first figure is the single step and the second is the sliding rollout. The color scale is logarithmic and shared by the panels of one figure.

In [ ]:
for field, title in MATRIX_FIELDS.items():
    if not matrices:
        break
    relative = {
        name: (matrix[field] / matrix[f"{field}_identity"]).mean(axis=1)
        for name, matrix in matrices.items()
    }
    norm = LogNorm(
        vmin=min(values.min() for values in relative.values()),
        vmax=max(values.max() for values in relative.values()),
    )
    fig, axes = plt.subplots(
        1,
        len(relative),
        figsize=(2.5 * len(relative) + 0.6, 2.7),
        constrained_layout=True,
        squeeze=False,
    )
    for ax, (name, values) in zip(axes[0], relative.items()):
        image = ax.imshow(values, cmap=ORDERED, norm=norm)
        matrix = matrices[name]
        ax.set_xticks(range(len(matrix["tested_at"])), matrix["tested_at"])
        ax.set_yticks(range(len(matrix["trained_at"])), matrix["trained_at"])
        ax.set_xlabel("Test context length")
        ax.set_title(MATRIX_SETS[name], fontsize="medium")
        ax.grid(False)
        for (i, j), value in np.ndenumerate(values):
            # Dark cells get light text and light cells get dark text.
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize="small",
                color="white" if norm(value) > 0.55 else INK,
            )
    axes[0, 0].set_ylabel("Training context length")
    fig.colorbar(
        image, ax=axes, label=f"{title}: error relative\nto the identity baseline"
    )
    plt.show()

### The diagonal

The error of every model at its own training length, against that length. Each line is one test set, and the sine bins go from light to dark with |b|. The bars are the standard deviation over the seeds.

In [ ]:
if matrices:
    sine_colors = dict(
        zip([f"sine_rescaled_{b}_grf" for b in SINE_BINS], ordered_colors(4))
    )
    fig, axes = plt.subplots(
        1,
        len(MATRIX_FIELDS),
        figsize=(4.4 * len(MATRIX_FIELDS), 3.0),
        constrained_layout=True,
    )
    for ax, (field, title) in zip(axes, MATRIX_FIELDS.items()):
        for name, matrix in matrices.items():
            lengths = [k for k in matrix["trained_at"] if k in matrix["tested_at"]]
            diagonal = np.stack(
                [
                    matrix[field][
                        matrix["trained_at"].index(k), :, matrix["tested_at"].index(k)
                    ]
                    for k in lengths
                ]
            )
            ax.errorbar(
                lengths,
                diagonal.mean(axis=1),
                yerr=diagonal.std(axis=1),
                color=sine_colors.get(name, INK),
                marker="o",
                markersize=4,
                capsize=2,
                label=MATRIX_SETS[name],
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xticks(lengths, lengths)
        ax.minorticks_off()
        ax.set_xlabel("Context length")
        ax.set_ylabel("Relative $\\ell^2$ error")
        ax.set_title(title, fontsize="medium")
    axes[-1].legend(fontsize="small", loc="center left", bbox_to_anchor=(1.02, 0.5))
    plt.show()

## Error against the CFL number (E2)

The submitted checkpoints of the three models at a context of 20 frames, on the in-distribution set. The CFL number of a segment is the largest |f'(u)| over the interval of states of its last context frame, times dt / dx. The bins hold equally many segments. Many segments of one coefficient draw can share one CFL number, so a bin can be empty, and an empty bin is not drawn.

The left panel is the single-step error relative to the identity baseline. The right panel is the error of the identity baseline itself, which shows how much moves in one step at each CFL number.

In [ ]:
NUM_CFL_BINS = 12

f = open_result("context_length", "cubic_1d_cfl")
if f is not None:
    with f:
        cfl = f["cfl"][()].reshape(-1)
        identity = f["baselines/identity/l2_onestep"][()]
        # The preset evaluates one test context length, the second axis.
        errors = {group: f[f"models/{group}/l2_onestep"][:, 0] for group in f["models"]}
    bin_indices, edges = equal_count_bins(cfl, NUM_CFL_BINS)
    centers = bin_centers(cfl, bin_indices, NUM_CFL_BINS)
    print(f"CFL numbers from {edges[0]:.2f} to {edges[-1]:.2f} on {cfl.size} segments")

    fig, (ax, ax_identity) = plt.subplots(
        1, 2, figsize=(7.4, 3.0), constrained_layout=True
    )
    for group, values in errors.items():
        relative, counts = binned_ratio(
            values, np.broadcast_to(identity, values.shape), bin_indices, NUM_CFL_BINS
        )
        is_filled = counts > 0
        ax.errorbar(
            centers[is_filled],
            relative.mean(axis=0)[is_filled],
            yerr=relative.std(axis=0)[is_filled],
            color=COLORS.get(group, INK),
            marker="o",
            markersize=4,
            capsize=2,
            label=LABELS.get(group, group),
        )
    identity_means, counts = mean_by_bin(
        identity.reshape(-1), bin_indices, NUM_CFL_BINS
    )
    ax_identity.plot(
        centers[counts > 0],
        identity_means[counts > 0],
        color=MUTED,
        marker="o",
        markersize=4,
    )
    for axis in (ax, ax_identity):
        plain_log_x(axis)
        axis.set_yscale("log")
        axis.set_xlabel("CFL number")
    ax.set_ylabel("Single-step error relative to\nthe identity baseline")
    ax.legend(fontsize="small")
    ax_identity.set_ylabel("Single-step error of\nthe identity baseline")
    plt.show()

## Flux refreshed every n steps (E3)

The flux networks are generated every n steps and held in between. n = 1 is the rollout of the submission, and n = 20 generates them once for the 20-step rollout. `l2_final` is the error of the last frame, which is the score of the response to Reviewer AyEF of 2026-11-12.

The right panel compares the trend with the numbers of that response, which are for 2D cubic in distribution. Both are divided by their value at n = 1, because the two sets have different error levels.

In [ ]:
# Final-frame relative l2 error on 2D cubic, in distribution, in units of 1e-1, from
# the response to Reviewer AyEF of 2026-11-12 (OpenReview forum Jv7ay9nWBN).
POSTED_2D = {1: 0.131, 2: 0.125, 4: 0.112, 8: 0.100}
REFRESH_FIELDS = ("l2_rollout", "l_inf_rollout", "l2_final", "l_inf_final")

f = open_result("flux_refresh", "cubic_1d")
if f is not None:
    with f:
        intervals = [int(n) for n in f.attrs["refresh_every"]]
        refresh = {
            field: mean_over_finite(f[f"models/{HFLUXNO}/{field}"][()], axis=(1, 2))
            for field in REFRESH_FIELDS
        }
    table = pd.DataFrame(
        {
            field: [mean_std(refresh[field][:, i]) for i in range(len(intervals))]
            for field in REFRESH_FIELDS
        },
        index=pd.Index(intervals, name="refresh every"),
    )
    display(table)

    fig, (ax, ax_ratio) = plt.subplots(
        1, 2, figsize=(7.4, 3.0), constrained_layout=True
    )
    for field, color in (("l2_rollout", COLORS[HFLUXNO]), ("l2_final", MUTED)):
        ax.errorbar(
            intervals,
            refresh[field].mean(axis=0),
            yerr=refresh[field].std(axis=0),
            color=color,
            marker="o",
            markersize=4,
            capsize=2,
            label={"l2_rollout": "All 20 frames", "l2_final": "Last frame"}[field],
        )
    ax.set_ylabel("Relative $\\ell^2$ error, 1D cubic")
    ax.legend(fontsize="small")

    if 1 in intervals:
        ratio = refresh["l2_final"] / refresh["l2_final"][:, [intervals.index(1)]]
        ax_ratio.errorbar(
            intervals,
            ratio.mean(axis=0),
            yerr=ratio.std(axis=0),
            color=COLORS[HFLUXNO],
            marker="o",
            markersize=4,
            capsize=2,
            label="1D cubic, this run",
        )
    ax_ratio.plot(
        list(POSTED_2D),
        np.asarray(list(POSTED_2D.values())) / POSTED_2D[1],
        color=INK,
        marker="s",
        markersize=4,
        linestyle="--",
        label="2D cubic, posted response",
    )
    ax_ratio.axhline(1.0, color=MUTED, linewidth=0.8)
    ax_ratio.set_ylabel("Last-frame error relative to n = 1")
    ax_ratio.legend(fontsize="small")
    for axis in (ax, ax_ratio):
        axis.set_xscale("log", base=2)
        axis.set_xticks(intervals, intervals)
        axis.minorticks_off()
        axis.set_xlabel("Steps per set of flux networks, n")
    plt.show()

## Shuffled, static and donor contexts (E4)

The flux networks are generated from another context than the true one, and the true last context frame is advanced with them.

- `true`: the context of the segment.
- `shuffled`: the same frames in a random order.
- `static`: copies of the last context frame.
- `donor_same`: the context of another initial condition with the same coefficients.
- `donor_other`: the contexts of trajectories with other coefficients, pooled.

`l2_onestep` is the headline. `l2_rollout` is a 20-step rollout with the flux networks held, so its `true` row is not the rollout of the submission.

In [ ]:
f = open_result("context_interventions", "cubic_1d")
interventions = None
if f is not None:
    with f:
        names = [str(name) for name in f.attrs["interventions"]]
        interventions = {
            "names": names,
            "distance": f["flux_derivative_distance"][()],
            "identity": {
                field: f[f"baselines/identity/{field}"][()]
                for field in ("l2_onestep", "l2_rollout")
            },
            "errors": {
                field: f[f"models/{HFLUXNO}/{field}"][()]
                for field in ("l2_onestep", "l2_rollout")
            },
        }
    first_other = len(INTERVENTIONS) + 1
    rows = {name: [names.index(name)] for name in (*INTERVENTIONS, "donor_same")}
    rows["donor_other"] = list(range(first_other, len(names)))

    summary = {}
    for field, errors in interventions["errors"].items():
        identity = mean_over_finite(interventions["identity"][field], axis=(0, 1))
        true = mean_over_finite(errors[..., rows["true"]], axis=(1, 2, 3))
        for row, columns in rows.items():
            per_seed = mean_over_finite(errors[..., columns], axis=(1, 2, 3))
            summary[(row, field)] = per_seed
            summary[(row, f"{field} / identity")] = per_seed / identity
            summary[(row, f"{field} / true")] = per_seed / true
    columns = [key[1] for key in summary if key[0] == "true"]
    table = pd.DataFrame(
        [
            [
                mean_std(summary[(row, column)])
                if "/" not in column
                else f"{summary[(row, column)].mean():.2f}"
                for column in columns
            ]
            for row in rows
        ],
        index=pd.Index(list(rows), name="encoder input"),
        columns=columns,
    )
    display(table)

### Cost of each intervention

The single-step error relative to the identity baseline. At 1, the dashed line, the prediction is no better than holding the last context frame.

In [ ]:
if interventions is not None:
    order = list(rows)[::-1]
    values = np.stack([summary[(row, "l2_onestep / identity")] for row in order])
    fig, ax = plt.subplots(figsize=(4.6, 2.4), constrained_layout=True)
    ax.barh(
        order,
        values.mean(axis=1),
        xerr=values.std(axis=1),
        height=0.6,
        color=COLORS[HFLUXNO],
        error_kw={"capsize": 2, "ecolor": INK},
    )
    for position, value in enumerate(values.mean(axis=1)):
        ax.text(
            value, position, f"  {value:.2f}", va="center", fontsize="small", color=INK
        )
    ax.axvline(1.0, color=INK, linestyle="--", linewidth=0.8)
    ax.set_xlabel("Single-step error relative to the identity baseline")
    ax.grid(axis="y", visible=False)
    plt.show()

### Error against the distance of the donor flux

For a donor with other coefficients, the distance is ||f'_donor - f'_receiver|| / ||f'_receiver||, with L2 norms over the interval of states of the receiver's last context frame. The bins hold equally many donor contexts. The error is divided by the error with the true context on the same segments, so 1 means that the donor context costs nothing.

The square is the donor with the same coefficients, whose distance is zero. It is drawn at the left edge of the logarithmic axis.

In [ ]:
NUM_DISTANCE_BINS = 12

if interventions is not None:
    errors = interventions["errors"]["l2_onestep"]
    distance = interventions["distance"][..., 1:].reshape(-1)
    bin_indices, edges = equal_count_bins(distance, NUM_DISTANCE_BINS)
    centers = bin_centers(distance, bin_indices, NUM_DISTANCE_BINS)
    donor = errors[..., first_other:]
    reference = np.broadcast_to(errors[..., rows["true"]], donor.shape)
    relative, counts = binned_ratio(donor, reference, bin_indices, NUM_DISTANCE_BINS)
    is_filled = counts > 0

    fig, ax = plt.subplots(figsize=(4.2, 3.6), constrained_layout=True)
    ax.errorbar(
        centers[is_filled],
        relative.mean(axis=0)[is_filled],
        yerr=relative.std(axis=0)[is_filled],
        color=COLORS[HFLUXNO],
        marker="o",
        markersize=4,
        capsize=2,
        label="Donor with other coefficients",
    )
    same = summary[("donor_same", "l2_onestep / true")]
    left = np.nanmin(centers) / 2
    ax.errorbar(
        [left],
        [same.mean()],
        yerr=[same.std()],
        color=INK,
        marker="s",
        markersize=5,
        capsize=2,
        linestyle="none",
        label="Donor with the same coefficients",
    )
    ax.axhline(1.0, color=MUTED, linewidth=0.8)
    plain_log_x(ax)
    ax.set_yscale("log")
    ax.set_xlabel("Distance between the flux derivatives")
    ax.set_ylabel("Single-step error relative to\nthe true context")
    fig.legend(loc="outside lower center", fontsize="small")
    plt.show()

## Effective flux (E5)

A consistent numerical flux returns f(u) + c on a constant state u. The generated flux network is evaluated on constant states between the smallest and the largest value of a context, and its derivative with respect to the state is compared with the wave speed f'.

All mismatches are relative L2 distances over the states of one context.

- `derivative`: the inferred wave speed against f'.
- `real_stencil`: the same derivative taken on the last context frame, a cross-check.
- `spread`: how much the inferred wave speed varies over the interfaces.
- `box_cubic`: the best cubic of the training box against f'. It is the error of the best member of the training family.
- `to_box_cubic`: the inferred wave speed against that cubic.

In [ ]:
FLUX_SETS = {"cubic_1d_limit10": "Pilot", "cubic_1d": "In distribution"} | {
    f"sine_rescaled_{b}_grf": f"Sine, |b| in ({b.replace('-', ', ')}]"
    for b in SINE_BINS
}


def load_effective_flux(name: str) -> dict | None:
    f = open_result("effective_flux", name)
    if f is None:
        return None
    with f:
        group = f[f"models/{HFLUXNO}"]
        return {
            "coefficient_names": tuple(str(n) for n in f.attrs["coefficient_names"]),
            "coefficients": f["coefficients"][()],
            "states": f["states"][()],
            "derivative": group["derivative"][()],
            "box_cubic_derivative": f["box_cubic_derivative"][()]
            if "box_cubic_derivative" in f
            else None,
            "mismatches": {
                key: values[()] for key, values in group["mismatches"].items()
            },
        }


flux_results = {name: load_effective_flux(name) for name in FLUX_SETS}
flux_results = {
    name: result for name, result in flux_results.items() if result is not None
}
sine_results = {
    name: result
    for name, result in flux_results.items()
    if result["box_cubic_derivative"] is not None
}


def fraction_closer(result: dict) -> np.ndarray:
    """Return, per seed, the fraction of contexts on which the inferred wave speed is
    closer to f' than the best cubic of the training box."""
    model, box_cubic = (
        result["mismatches"]["derivative"],
        result["mismatches"]["box_cubic"],
    )
    is_finite = np.isfinite(model) & np.isfinite(box_cubic)
    return ((model < box_cubic) & is_finite).sum(axis=(1, 2)) / is_finite.sum(
        axis=(1, 2)
    )


def median_over_contexts(values: np.ndarray) -> np.ndarray:
    """Return the median over the contexts with a finite value, per seed."""
    flat = values.reshape(values.shape[0], -1)
    return np.asarray([np.median(row[np.isfinite(row)]) for row in flat])


summary_rows = {}
for name, result in flux_results.items():
    row = {
        key: f"{median_over_contexts(result['mismatches'][key]).mean():.3f}"
        if key in result["mismatches"]
        else ""
        for key in ("derivative", "real_stencil", "spread", "box_cubic", "to_box_cubic")
    }
    row["closer than box cubic"] = (
        f"{fraction_closer(result).mean():.3f} ± {fraction_closer(result).std():.3f}"
        if name in sine_results
        else ""
    )
    summary_rows[FLUX_SETS[name]] = row
if summary_rows:
    print("Median over the contexts, mean over the seeds")
    display(pd.DataFrame(summary_rows).T)

### Criterion of the pilot

The estimator on constant states is adopted if its mismatch in distribution is below the `box_cubic` mismatch of the sine bin (2, 3]. Above that value it could not resolve whether the model is closer to the sine flux than a member of the training family.

In [ ]:
pilot = flux_results.get("cubic_1d_limit10")
reference_bin = flux_results.get("sine_rescaled_2-3_grf")
if pilot is not None and reference_bin is not None:
    floor = median_over_contexts(pilot["mismatches"]["derivative"]).mean()
    effect = median_over_contexts(reference_bin["mismatches"]["box_cubic"]).mean()
    real = median_over_contexts(pilot["mismatches"]["real_stencil"]).mean()
    print(f"In distribution, constant states: {floor:.3f}")
    print(f"In distribution, real stencils:   {real:.3f}")
    print(f"Best cubic of the training box on (2, 3]: {effect:.3f}")
    print("The criterion is met." if floor < effect else "The criterion is not met.")

### Inferred wave speed on single contexts

Each column is one sine bin. The rows are the contexts at the 25th, 50th and 75th percentile of the `derivative` mismatch, so the middle row is a typical context and not a chosen one. The band around the inferred wave speed is the standard deviation over the seeds.

In [ ]:
PERCENTILES = (25, 50, 75)

if sine_results:
    fig, axes = plt.subplots(
        len(PERCENTILES),
        len(sine_results),
        figsize=(2.6 * len(sine_results), 1.9 * len(PERCENTILES)),
        constrained_layout=True,
        squeeze=False,
    )
    for column, (name, result) in enumerate(sine_results.items()):
        _, flux_derivative = FLUX_FAMILIES[result["coefficient_names"]]
        mismatch = result["mismatches"]["derivative"].mean(axis=0)
        ranks = np.argsort(mismatch, axis=None)
        for row, percentile in enumerate(PERCENTILES):
            ax = axes[row, column]
            trajectory, window = np.unravel_index(
                ranks[int(percentile / 100 * (ranks.size - 1))], mismatch.shape
            )
            states = result["states"][trajectory, window]
            inferred = result["derivative"][:, trajectory, window]
            ax.plot(
                states,
                flux_derivative(result["coefficients"][trajectory], states),
                color=INK,
                label="True $f'$",
            )
            ax.plot(
                states,
                result["box_cubic_derivative"][trajectory, window],
                color=MUTED,
                linestyle="--",
                label="Best cubic of the training box",
            )
            ax.plot(
                states, inferred.mean(axis=0), color=COLORS[HFLUXNO], label="Inferred"
            )
            ax.fill_between(
                states,
                inferred.mean(axis=0) - inferred.std(axis=0),
                inferred.mean(axis=0) + inferred.std(axis=0),
                color=COLORS[HFLUXNO],
                alpha=0.25,
                linewidth=0,
            )
            ax.set_title(
                f"{FLUX_SETS[name]}\n" * (row == 0) + f"{percentile}th percentile, "
                f"mismatch {mismatch[trajectory, window]:.2f}",
                fontsize="small",
            )
            if row == len(PERCENTILES) - 1:
                ax.set_xlabel("State $u$")
    for row_axes in axes:
        row_axes[0].set_ylabel("Wave speed")
    fig.legend(
        *axes[0, 0].get_legend_handles_labels(), loc="outside lower center", ncols=3
    )
    plt.show()

### Distribution of the mismatch

The share of contexts with a mismatch below the value on the horizontal axis, for the inferred wave speed and for the best cubic of the training box. Where the line of the model lies to the left of the dashed line, the model is closer to f' than a member of the training family.

In [ ]:
def ecdf(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    values = np.sort(values[np.isfinite(values)])
    return values, np.arange(1, values.size + 1) / values.size


if sine_results:
    fig, axes = plt.subplots(
        1,
        len(sine_results),
        figsize=(2.6 * len(sine_results), 2.5),
        sharey=True,
        constrained_layout=True,
        squeeze=False,
    )
    for ax, (name, result) in zip(axes[0], sine_results.items()):
        ax.plot(
            *ecdf(result["mismatches"]["derivative"].mean(axis=0)),
            color=COLORS[HFLUXNO],
            label="Inferred",
        )
        ax.plot(
            *ecdf(result["mismatches"]["box_cubic"][0]),
            color=MUTED,
            linestyle="--",
            label="Best cubic of the training box",
        )
        ax.set_xscale("log")
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set_xlabel("Mismatch with $f'$")
        ax.set_title(FLUX_SETS[name], fontsize="medium")
    axes[0, 0].set_ylabel("Share of contexts")
    fig.legend(
        *axes[0, 0].get_legend_handles_labels(), loc="outside lower center", ncols=2
    )
    plt.show()

### Does the model do better where the flux is far from every cubic?

Each context has two mismatches with f': that of the inferred wave speed and that of the best cubic of the training box. The upper panels show how the contexts are spread over the two. Below the diagonal the model is closer to f' than the cubic. The lower panels show the share of such contexts in bins of the `box_cubic` mismatch, with the standard deviation over the seeds. A share that rises to the right would mean that the model goes beyond the training family where a cubic cannot follow the flux.

In [ ]:
NUM_BOX_BINS = 10

if sine_results:
    fig, axes = plt.subplots(
        2,
        len(sine_results),
        figsize=(2.6 * len(sine_results), 4.8),
        constrained_layout=True,
        squeeze=False,
    )
    for column, (name, result) in enumerate(sine_results.items()):
        model = result["mismatches"]["derivative"]
        box_cubic = result["mismatches"]["box_cubic"][0]
        x, y = box_cubic.reshape(-1), model.mean(axis=0).reshape(-1)
        is_valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        low = min(x[is_valid].min(), y[is_valid].min())
        high = max(x[is_valid].max(), y[is_valid].max())
        edges = np.geomspace(low, high, 60)

        ax = axes[0, column]
        ax.hist2d(
            x[is_valid], y[is_valid], bins=(edges, edges), cmap=ORDERED, norm=LogNorm()
        )
        ax.plot([low, high], [low, high], color=INK, linewidth=0.8, linestyle="--")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_minor_formatter(NullFormatter())
        ax.set_title(FLUX_SETS[name], fontsize="medium")
        ax.grid(False)

        bin_indices, _ = equal_count_bins(x, NUM_BOX_BINS)
        centers = bin_centers(x, bin_indices, NUM_BOX_BINS)
        is_closer = (model < box_cubic).astype(float).reshape(model.shape[0], -1)
        share, counts = mean_by_bin(is_closer, bin_indices, NUM_BOX_BINS)
        is_filled = counts > 0
        ax = axes[1, column]
        ax.errorbar(
            centers[is_filled],
            share.mean(axis=0)[is_filled],
            yerr=share.std(axis=0)[is_filled],
            color=COLORS[HFLUXNO],
            marker="o",
            markersize=4,
            capsize=2,
        )
        ax.axhline(0.5, color=MUTED, linewidth=0.8)
        ax.set_xscale("log")
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set_ylim(0, 1)
        ax.set_xlabel("Mismatch of the best cubic\nof the training box")
    axes[0, 0].set_ylabel("Mismatch of the\ninferred wave speed")
    axes[1, 0].set_ylabel("Share of contexts with\nthe model closer to $f'$")
    plt.show()

## One step of another size (E6)

The context is sampled at the training step and encoded once. The generated flux networks then advance one step of half, one and two training steps. The error is relative to the identity baseline of the same step size, and the CFL number is that of the step taken.

The bins of the CFL number are shared by the step sizes. Where the lines lie on each other, the error depends on the CFL number alone. A gap between them at one CFL number is the error of using the flux at another time step than it was trained on. A point is drawn only if its bin holds at least `MIN_SEGMENTS` segments of that step size.

In [ ]:
NUM_STEP_BINS = 16
MIN_SEGMENTS = 50

f = open_result("time_step", "cubic_1d")
if f is not None:
    with f:
        step_frames = [int(m) for m in f.attrs["step_frames"]]
        frames_per_step = int(f.attrs["frames_per_step"])
        cfl = f["cfl"][()]
        errors = f[f"models/{HFLUXNO}/l2"][()]
        identity = f[f"models/{HFLUXNO}/l2_identity"][()]
    labels = [f"{m / frames_per_step:g} $\\Delta t$" for m in step_frames]
    bin_indices, edges = equal_count_bins(cfl.reshape(-1), NUM_STEP_BINS)
    bin_indices = bin_indices.reshape(cfl.shape)
    centers = bin_centers(cfl.reshape(-1), bin_indices.reshape(-1), NUM_STEP_BINS)

    overall = {
        label: mean_over_finite(errors[..., i], axis=(1, 2))
        / mean_over_finite(identity[..., i], axis=(1, 2))
        for i, label in enumerate(labels)
    }
    display(
        pd.DataFrame(
            {
                label: [f"{ratio.mean():.3f} ± {ratio.std():.3f}"]
                for label, ratio in overall.items()
            },
            index=["error relative to the identity baseline, all segments"],
        )
    )

    fig, ax = plt.subplots(figsize=(4.4, 3.0), constrained_layout=True)
    for i, (label, color) in enumerate(zip(labels, ordered_colors(len(labels)))):
        relative, counts = binned_ratio(
            errors[..., i],
            identity[..., i],
            bin_indices[..., i].reshape(-1),
            NUM_STEP_BINS,
        )
        is_drawn = counts >= MIN_SEGMENTS
        if not is_drawn.any():
            print(f"{label}: no bin holds {MIN_SEGMENTS} segments.")
            continue
        ax.errorbar(
            centers[is_drawn],
            relative.mean(axis=0)[is_drawn],
            yerr=relative.std(axis=0)[is_drawn],
            color=color,
            marker="o",
            markersize=4,
            capsize=2,
            label=label,
        )
    plain_log_x(ax)
    ax.set_yscale("log")
    ax.set_xlabel("CFL number of the step")
    ax.set_ylabel("Single-step error relative to\nthe identity baseline")
    ax.legend(title="Step size", fontsize="small")
    plt.show()